# Notebook 3 — Validación FSM v6

**Inputs:** `outputs/trajectories_*.csv`, `outputs/clusters_k4.csv`, `data/grades_export_anon.csv`  
**Outputs:** `outputs/fsm_v6_transition_matrix.csv`, `outputs/fsm_v6_metrics_per_student.csv`

In [ ]:
import os, numpy as np, random, pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
BASE_PATH = '.'
INPUTS  = os.path.join(BASE_PATH, 'data')
OUTPUTS = os.path.join(BASE_PATH, 'outputs')
ID_COL = 'email'; STATE_COL = 'state'; TIME_COL = 'timestamp'
print(f'INPUTS: {INPUTS} | OUTPUTS: {OUTPUTS}')


In [ ]:
traj_full    = pd.read_csv(f"{OUTPUTS}/trajectories_full.csv")
traj_summary = pd.read_csv(f"{OUTPUTS}/trajectories_summary.csv")
clusters     = pd.read_csv(f"{OUTPUTS}/clusters_k4.csv")
grades       = pd.read_csv(f"{INPUTS}/grades_export_anon.csv")
grades.columns = [c.strip().lower() for c in grades.columns]
print('Columnas grades:', list(grades.columns))
print(f'Estados FSM: {traj_full[STATE_COL].unique()}')
print(f'Alumnos: {traj_full[ID_COL].nunique()} | Eventos totales: {len(traj_full)}')


## Distribución de estados FSM v6

In [ ]:
state_freq = traj_full[STATE_COL].value_counts(normalize=True)
fig, ax = plt.subplots(figsize=(7,4))
state_freq.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Distribución de estados FSM v6')
ax.set_ylabel('Proporción'); ax.set_xlabel('Estado')
plt.xticks(rotation=0); plt.tight_layout()
plt.savefig(f"{OUTPUTS}/fig1_state_distribution.png", dpi=150); plt.close()
print(state_freq.round(3))


## Matriz de transición FSM v6

In [ ]:
traj_full = traj_full.sort_values([ID_COL, TIME_COL])
traj_full['prev_state'] = traj_full.groupby(ID_COL)[STATE_COL].shift(1)
T = pd.crosstab(traj_full['prev_state'], traj_full[STATE_COL], normalize='index')
fig, ax = plt.subplots(figsize=(7,5))
sns.heatmap(T, annot=True, cmap='Blues', fmt='.2f', ax=ax, linewidths=0.5)
ax.set_title('Matriz de transición FSM v6')
plt.tight_layout()
plt.savefig(f"{OUTPUTS}/fig2_transition_matrix.png", dpi=150); plt.close()
print(T.round(3))


## Métricas FSM por alumno + Correlaciones Spearman

In [ ]:
fsm_metrics = traj_full.groupby([ID_COL, STATE_COL]).size().unstack(fill_value=0)
fsm_metrics['total'] = fsm_metrics.sum(axis=1)
for c in list(fsm_metrics.columns):
    if c != 'total': fsm_metrics[f'{c}_prop'] = fsm_metrics[c] / fsm_metrics['total']
fsm_metrics = fsm_metrics.reset_index()

GRADE_COL = next((c for c in ['nota_final','nota','final_grade'] if c in grades.columns), None)
print(f'Columna de nota: {GRADE_COL}')
fsm_gr = fsm_metrics.merge(grades[[ID_COL, GRADE_COL]], on=ID_COL)
print(f'\nCorrelaciones Spearman FSM vs {GRADE_COL} (n={len(fsm_gr)}):')
for s in ['NAV','REC','PRACT','EVAL']:
    if f'{s}_prop' in fsm_gr.columns:
        rho, p = spearmanr(fsm_gr[f'{s}_prop'], fsm_gr[GRADE_COL])
        sig = '**' if p<0.05 else ('*' if p<0.10 else 'ns')
        print(f'  {s}: rho={rho:+.3f}, p={p:.4f}  {sig}')


In [ ]:
T.to_csv(f"{OUTPUTS}/fsm_v6_transition_matrix.csv")
fsm_metrics.to_csv(f"{OUTPUTS}/fsm_v6_metrics_per_student.csv", index=False)
print('Outputs guardados.')
